# MIORPA 2026: pluralistic activation steering

This notebook nudges small language models to give answers that hold more than one cultural viewpoint, instead of defaulting to just one. It does this while the model is generating text, by adding a small vector to its internal activations. No retraining involved.

Before running anything, set the runtime to a GPU (Runtime > Change runtime type).

Steps below:

1. Build calibration pairs from PRISM, real human ratings. One set is answers two demographic groups both liked, the other is answers only one group liked.
2. Build test questions the model hasn't seen before.
3. Get the steering vector from those pairs, 4 different ways, for each model.
4. Generate answers with and without steering.
5. Score the answers automatically, then judge whether steering actually made them more pluralistic.

Everything saves as it goes. If Colab disconnects, just run again and it picks up where it stopped instead of starting over.

## Step 1: environment

In [ ]:
!pip -q install transformers accelerate sentence-transformers bert-score scikit-learn pandas datasets huggingface_hub openpyxl

import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Set Runtime > Change runtime type > GPU before continuing.')

In [ ]:
import os, sys, glob

# mount drive if we're on colab and it isn't mounted yet
try:
    import google.colab  # noqa: F401
    if not os.path.isdir('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
except ImportError:
    pass

# look for the folder that actually has the miorpa package in it
candidate_paths = [
    '/content/drive/MyDrive/MIORPA PROJECT',
    '/content/drive/MyDrive/MIORPA_PROJECT',
    '/content/MIORPA PROJECT',
    '/content/miorpa_project',
    os.getcwd(),
]
candidate_paths += [os.path.dirname(p) for p in glob.glob('/content/**/miorpa/config.py', recursive=True)]
candidate_paths += [os.path.dirname(p) for p in
                     glob.glob('/content/drive/MyDrive/**/miorpa/config.py', recursive=True)]

project_dir = None
for path in candidate_paths:
    if path and os.path.isfile(os.path.join(path, 'miorpa', 'config.py')):
        project_dir = path
        break

if project_dir is None:
    print('Could not find the miorpa package anywhere.\n')
    print('The code needs to get into this runtime. Pick one:\n')
    print('  A. Upload the whole "MIORPA PROJECT" folder to the top level of')
    print('     your Google Drive, then re-run this cell.\n')
    print('  B. Zip the miorpa folder, upload it here, unzip it:')
    print('        from google.colab import files; files.upload()')
    print('        !unzip -q miorpa.zip -d /content/\n')
    print('The dataset does not need uploading, it downloads from Hugging')
    print('Face automatically. Only the code has to be here.')
    raise SystemExit('miorpa package not found')

os.chdir(project_dir)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

print('working in', project_dir)
print('package present:', os.path.isdir('miorpa'))
if project_dir.startswith('/content/drive/'):
    print('results are saving to Google Drive - safe if this session drops')
else:
    print('WARNING: results are saving to the temporary Colab disk, not Drive.')
    print('They will be lost if this session disconnects. Put the project in')
    print('MyDrive and re-run this cell, or download from Step 13 before ending.')

dataset_files = {name: os.path.isfile(f'dataset/{name}.jsonl') for name in
                  ('survey', 'conversations', 'utterances')}
print('dataset present:', all(dataset_files.values()), dataset_files)
if not all(dataset_files.values()):
    print('\nmissing PRISM files get downloaded from Hugging Face in Step 1b')

In [ ]:
from miorpa import config as cfg
from miorpa import run_pipeline as rp
from miorpa import data, vectors, steering, evaluate, benchmarks, judge

import importlib
for module in (cfg, data, vectors, steering, evaluate, benchmarks, judge, rp):
    importlib.reload(module)

print('models available:')
for model in cfg.MODELS:
    print(f'  {model.short_name:16s} {model.params:>5s}  {model.hf_id}')

print(f'\ngeneration batch size, picked from the attached GPU: {cfg.GENERATION_BATCH_SIZE}')

## Step 1b: dataset

Downloads PRISM (about 128 MB) from Hugging Face if it isn't already on disk.

If it's gated for your account: accept the terms on the dataset page, then uncomment and run the login line below.

In [ ]:
# if PRISM is gated for your account, uncomment and run this once:
# from huggingface_hub import notebook_login; notebook_login()

dataset_ready = data.ensure_dataset()
print('\ndataset ready:', dataset_ready)

## Step 2: calibration pairs

For each axis (origin, religion, age) this finds pairs of answers to a similar question: one both sides rated highly (balanced), one only one side rated highly (one-sided). That pair is what the steering vector gets built from later.

Questions are matched by exact text first, then by meaning (cosine similarity 0.75 or above) for whatever's left. That second pass is what makes a thin axis like age usable at all.

Two sanity checks run after: is one side's answers systematically longer than the other (a possible confound), and does the split just track which AI model wrote the answer instead of which group liked it.

In [ ]:
pairs = rp.stage_pairs()

for axis, pairs_for_axis in pairs.items():
    if len(pairs_for_axis):
        print(f"\n{axis}: {len(pairs_for_axis):,} pairs")
        print(pairs_for_axis['match_type'].value_counts().to_string())

print('\n' + '=' * 60)
print('CONFOUND CHECK')
print('=' * 60)
for axis, pairs_for_axis in pairs.items():
    data.diagnose_pairs(pairs_for_axis, axis)

print('\n' + '=' * 60)
print('MODEL CLUSTERING')
print('=' * 60)
for axis, pairs_for_axis in pairs.items():
    data.diagnose_model_clustering(pairs_for_axis, axis)

# if any axis looks thin, this probably loaded an old cached file.
# force a rebuild with:
#   pairs = rp.stage_pairs(force=True)

## Step 3: test questions

Builds the questions used later to check whether steering worked. Pulled from the World Values Survey and Anthropic's persona evaluations.

Any test question too close to a calibration question from Step 2 (cosine 0.50 or above) gets removed. Otherwise the model could just be repeating what it saw during calibration instead of generalising to something new.

In [ ]:
eval_sets = rp.stage_benchmarks(per_axis=cfg.EVAL_QUESTIONS_PER_AXIS)

for axis, questions in eval_sets.items():
    print(f"\n{axis} ({len(questions)} questions), first 3:")
    for question in questions[:3]:
        print('  -', question)

## Step 4: steering vectors

This builds the actual steering vector: the direction inside the model's activations that stands for "pluralism" on a given axis. Four ways to compute it, per model per axis (full detail in miorpa/README.md):

- MoD, mean difference between balanced and one-sided activations. Simplest, and the one the reference paper (Im and Li, 2026) found works best.
- PoD and PoE, two PCA-based variants.
- CoE, a linear classifier boundary between the two sets.

Plus a random vector of the same size, as a control. If a random direction moves the output as much as a real one, the effect isn't real.

Injected 45% of the way through the network, based on the reference paper's own tuned result. Step 9's layer sweep checks whether that holds for our models too.

Runs across all six models listed in the next cell, one at a time.

In [ ]:
# All six small models, smallest first. Each one runs completely - vectors,
# generation, evaluation, then both judges - before the next one starts, so
# a disconnect only costs whichever model was in progress, not the set.
MODELS_TO_RUN = [
    'SmolLM2-135M',
    'Qwen2.5-0.5B',
    'SmolLM2-360M',
    'Qwen2.5-1.5B',
    'SmolLM2-1.7B',
    'Qwen2.5-3B',
]
print('running, in order:', MODELS_TO_RUN)

## Step 5: smoke test

Runs one question through the model twice, once plain and once steered. Read both answers before running the full batch. If something's broken here, it's broken 30,000 times over downstream.

In [ ]:
spec = cfg.MODELS_BY_NAME[MODELS_TO_RUN[0]]
smoke_bundle = rp.stage_vectors(pairs, model_names=[spec.short_name])[spec.short_name]
axis = 'origin'

model, tokenizer, device = vectors.load_model(spec)
layer = smoke_bundle['layer']
steering_vector = vectors.get_vector(smoke_bundle, axis, 'MoD')

test_question = ['Should economic growth be prioritised over environmental protection?']

baseline_answer = steering.generate(model, tokenizer, test_question, device, layer_idx=layer, vector=None, alpha=0.0)
steered_answer = steering.generate(model, tokenizer, test_question, device, layer_idx=layer, vector=steering_vector, alpha=cfg.DEFAULT_ALPHA)

print('QUESTION:', test_question[0])
print('\n--- BASELINE ---')
print(baseline_answer[0])
print(f'\n--- STEERED (MoD, alpha={cfg.DEFAULT_ALPHA}, layer {layer}) ---')
print(steered_answer[0])

vectors.free(model)

## Step 6-7-11: generation, evaluation, and judging - per model

Runs every model in `MODELS_TO_RUN` completely, one at a time: builds its steering vectors, generates every condition, scores with BERTScore/perplexity, then judges every pair with both Mistral and Prometheus. Results save to Drive as each stage finishes.

Safe to stop and restart: generation checkpoints per condition, judging checkpoints per batch. Re-running skips whatever is already done, model by model - a finished model is never redone just because a later one hasn't started yet.

In [ ]:
import pandas as pd

all_scored, all_summary, all_pairs = [], [], []
all_resolved_mistral, all_resolved_prometheus = [], []

for model_name in MODELS_TO_RUN:
    print('\n' + '#' * 70)
    print(f'# {model_name}')
    print('#' * 70)

    spec = cfg.MODELS_BY_NAME[model_name]
    bundle = rp.stage_vectors(pairs, model_names=[model_name])

    run_cfg = cfg.RunConfig(
        models=[model_name],
        axes=cfg.AXES,
        methods=('MoD', 'PoD', 'PoE', 'CoE'),
        alphas=(cfg.DEFAULT_ALPHA,),
        questions_per_axis=cfg.EVAL_QUESTIONS_PER_AXIS,
        include_random_control=True,
        tag=f'main_{spec.slug}',
    )
    print(run_cfg.describe())

    rp.stage_generate(run_cfg, eval_sets, bundle)
    scored_m, summary_m = rp.stage_evaluate(run_cfg)
    all_scored.append(scored_m)
    all_summary.append(summary_m)

    pairs_m = judge.build_pairwise_set(scored_m, tag=run_cfg.tag)
    # pair_id resets to P000000 for every model's own build_pairwise_set
    # call, so without this prefix two different models could share the
    # same pair_id and get silently merged together once everything below
    # is concatenated into one combined table.
    pairs_m['pair_id'] = spec.slug + '_' + pairs_m['pair_id'].astype(str)
    all_pairs.append(pairs_m)

    judged_mistral_m = judge.run_judge(pairs_m, "mistral", tag=run_cfg.tag)
    resolved_mistral_m = judge.resolve_pairwise(judged_mistral_m, tag=run_cfg.tag)
    judge.unload_judge()

    judged_prometheus_m = judge.run_judge(pairs_m, "prometheus", tag=run_cfg.tag)
    resolved_prometheus_m = judge.resolve_pairwise(judged_prometheus_m, tag=run_cfg.tag)
    judge.unload_judge()

    all_resolved_mistral.append(resolved_mistral_m)
    all_resolved_prometheus.append(resolved_prometheus_m)

    print(f'\n--- {model_name} win rates (mistral) ---')
    display(judge.summarise_pairwise(resolved_mistral_m))
    print(f'\n=== {model_name} finished, results saved ===\n')

scored = pd.concat(all_scored, ignore_index=True)
summary = pd.concat(all_summary, ignore_index=True)
pairs_judged = pd.concat(all_pairs, ignore_index=True)
resolved_mistral = pd.concat(all_resolved_mistral, ignore_index=True)
resolved_prometheus = pd.concat(all_resolved_prometheus, ignore_index=True)

scored.to_csv(cfg.EVAL_DIR / 'scored_main.csv', index=False, encoding='utf-8')
summary.to_csv(cfg.EVAL_DIR / 'summary_main.csv', index=False, encoding='utf-8')

run = cfg.RunConfig(models=MODELS_TO_RUN, tag='main')
print('\nall models finished. combined tables: scored, summary, resolved_mistral, resolved_prometheus')

In [ ]:
# generation now happens inside the per-model loop above.
print('generation completed above, per model')

## Step 7: evaluation - already run above

Scored inside the per-model loop above, right after generation. `summary` below is the combined table across all six models. Two automatic metrics:

| metric | checks | good direction |
|---|---|---|
| bertscore | did the answer stay on topic | higher |
| perplexity | is the answer still fluent English | lower |

Pluralism itself isn't scored here, that needs a judge - also already run above, in the same loop.

In [ ]:
# evaluation now happens inside the per-model loop above; scored and
# summary here are already the combined tables across every model.
summary

## Step 8: read the result

This only shows whether steering broke anything, not whether it worked. If bertscore drops or perplexity climbs sharply for a condition, that model/axis/method combination is degrading, and no judge score later will fix that.

Whether pluralism actually improved needs the judge results from Step 11.

In [ ]:
import pandas as pd

pivot = summary.pivot_table(
    index=['model', 'axis'],
    columns='method',
    values=['bertscore', 'perplexity', 'n_words'],
)
display(pivot.round(3))

baseline_rows = summary[summary['method'] == 'baseline'].set_index(['model', 'axis'])
steered_rows = summary[summary['method'] != 'baseline']

deltas_list = []
for _, row in steered_rows.iterrows():
    key = (row['model'], row['axis'])
    if key not in baseline_rows.index:
        continue
    baseline = baseline_rows.loc[key]
    deltas_list.append({
        'model': row['model'], 'axis': row['axis'], 'method': row['method'],
        'bertscore_change': row['bertscore'] - baseline['bertscore'],
        'perplexity_ratio': row['perplexity'] / baseline['perplexity'] if baseline['perplexity'] else float('nan'),
        'empty_rate': row['empty_rate'],
    })

deltas = pd.DataFrame(deltas_list).sort_values('bertscore_change', ascending=False)
print('\nchange against each model/axis baseline:')
display(deltas.round(3))

degraded = deltas[(deltas['bertscore_change'] < -0.05)
                   | (deltas['perplexity_ratio'] > 1.5)
                   | (deltas['empty_rate'] > 0.05)]
if len(degraded):
    print('\nsteering degraded the output here, lower alpha for these conditions:')
    display(degraded.round(3))
else:
    print('\nno condition degraded bertscore or perplexity noticeably')

if 'Random' in set(summary['method']):
    print('\nrandom control vs MoD (automatic metrics only):')
    display(summary[summary['method'].isin(['baseline', 'MoD', 'Random'])]
            [['model', 'axis', 'method', 'bertscore', 'perplexity', 'n_words']].round(3))

## Step 8b: is the axis gap real, or just a scaling artefact?

The main run gave every axis the same alpha. But alpha only multiplies the vector, and the vectors are not the same length: on Qwen2.5-1.5B, |MoD| is 14 for origin, 40 for age and 85 for religion. What the model actually receives is alpha times that length, so religion was pushed about six times harder than origin. The axes were never compared at the same strength.

Two cells here. The first measures how big that push is next to the hidden state it is added to, which is what says whether the model was nudged or drowned. The second runs the naive setup and the corrected one back to back, so the difference between them is measured rather than assumed.

In [ ]:
import pandas as pd

# |alpha * v| means nothing without something to compare it against. This
# reports it as a percentage of the typical |h| at the injection layer.
ratio_frames = []
for model_name in MODELS_TO_RUN:
    print(f'\n--- push ratio: {model_name} ---')
    ratio_frames.append(rp.measure_push_ratio(model_name, pairs).assign(model=model_name))

push_ratio = pd.concat(ratio_frames, ignore_index=True)
display(push_ratio)

In [ ]:
import pandas as pd

# Same model, same questions, two conditions: one shared alpha for every
# axis, then alpha rescaled per axis so the push is equal everywhere.
comparison_frames = []
for model_name in MODELS_TO_RUN:
    print(f'\n--- push comparison: {model_name} ---')
    comparison_frames.append(rp.ablation_equalised_push(model_name).assign(model_run=model_name))

push_comparison = pd.concat(comparison_frames, ignore_index=True)

steered = push_comparison[push_comparison['method'] == 'MoD']
display(steered.pivot_table(index=['model_run', 'axis'], columns='push_mode',
                            values=['bertscore', 'perplexity']).round(3))

print('\nIf religion and age recover once the push is equalised, the gap in the')
print('main run was a scaling artefact. If they stay flat at a strength that')
print('leaves origin coherent, the axes genuinely differ and that is the finding.')

## Step 9: ablations

Three checks, run on every model in `MODELS_TO_RUN`, one at a time:

- Random control: does a random vector move the output as much as the real one? If yes, the effect isn't real.
- Alpha sweep: how strong can steering get before the model breaks?
- Layer sweep: which layer actually carries the cultural signal, and does that answer change with model size?

Without a judge these only show whether the vector perturbs the model differently from noise, not whether it perturbs it toward pluralism. That needs the judge scores from the step above. Cheap relative to the main run - roughly 15 minutes per model for all three checks combined - so running the full set is worth it rather than guessing from one model.

In [ ]:
import pandas as pd

# ablations now run on every model in MODELS_TO_RUN, not just one - cheap
# enough relative to the main run that there's no reason to guess from a
# single model's curve.
all_control_scored, all_control_summary = [], []
for model_name in MODELS_TO_RUN:
    print(f'\n--- random control: {model_name} ---')
    control_scored_m, control_summary_m = rp.ablation_random_control(model_name)
    all_control_scored.append(control_scored_m)
    all_control_summary.append(control_summary_m)

control_scored = pd.concat(all_control_scored, ignore_index=True)
control_summary = pd.concat(all_control_summary, ignore_index=True)

In [ ]:
import pandas as pd

all_alpha_scored, all_alpha_summary = [], []
for model_name in MODELS_TO_RUN:
    print(f'\n--- alpha sweep: {model_name} ---')
    alpha_scored_m, alpha_summary_m = rp.ablation_alpha_sweep(model_name)
    all_alpha_scored.append(alpha_scored_m)
    all_alpha_summary.append(alpha_summary_m)

alpha_scored = pd.concat(all_alpha_scored, ignore_index=True)
alpha_summary = pd.concat(all_alpha_summary, ignore_index=True)

curve = alpha_summary.groupby(['model', 'alpha'])[['bertscore', 'perplexity', 'n_words', 'empty_rate']].mean()
display(curve.round(3))

print('\nThis curve gives the upper bound on alpha, not the optimum, per model.')
print('Where perplexity starts climbing and bertscore starts falling is where')
print('that model is breaking. The best alpha below that line is a pluralism')
print('question, which needs the judge scores from the step above.')

In [ ]:
import pandas as pd

all_layer_summary = []
for model_name in MODELS_TO_RUN:
    print(f'\n--- layer sweep: {model_name} ---')
    layer_summary_m = rp.ablation_layer_sweep(model_name)
    all_layer_summary.append(layer_summary_m)

layer_summary = pd.concat(all_layer_summary, ignore_index=True)
display(layer_summary.groupby(['model', 'layer_fraction'])[['bertscore', 'perplexity', 'n_words']].mean().round(3))

print('\nIm and Li steer Llama-2-7b-chat at layer 13 of 32, about 0.41 depth.')
print('config.LAYER_DEPTH_FRACTION is set to 0.45 on that basis. This sweep')
print('checks whether that holds across model sizes here too, not just one.')

## Step 10: export

Saves everything to `MIORPA_results.xlsx` - generations, scores, and judge win rates across all six models - for the paper and the shared sheet.

In [ ]:
import pandas as pd

output_path = cfg.RESULTS_DIR / 'MIORPA_results.xlsx'

winrates_mistral = judge.summarise_pairwise(resolved_mistral)
winrates_prometheus = judge.summarise_pairwise(resolved_prometheus)

optional_sheets = {
    'random_control': globals().get('control_summary'),
    'alpha_sweep': globals().get('alpha_summary'),
    'layer_sweep': globals().get('layer_summary'),
    'push_ratio': globals().get('push_ratio'),
    'push_comparison': globals().get('push_comparison'),
}

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    summary.to_excel(writer, sheet_name='main_summary', index=False)
    deltas.to_excel(writer, sheet_name='deltas_vs_baseline', index=False)
    scored.head(3000).to_excel(writer, sheet_name='generations_sample', index=False)
    winrates_mistral.to_excel(writer, sheet_name='judge_winrates_mistral', index=False)
    winrates_prometheus.to_excel(writer, sheet_name='judge_winrates_prometheus', index=False)
    for sheet_name, sheet_data in optional_sheets.items():
        if sheet_data is not None:
            sheet_data.to_excel(writer, sheet_name=sheet_name, index=False)
        else:
            print(f'{sheet_name}: not run yet, skipped')

print('saved', output_path)

## Step 11: pairwise LLM judging - already run above

Judging happened inside the per-model loop above, one model at a time, so a disconnect only costs the model in progress rather than the whole judged set. The cells below just show the combined results across every model that has finished so far.

This is the part the automatic metrics can't do: is the steered answer actually more pluralistic than the baseline, not just still coherent.

Two judges, neither from a model family under test (three of the six models are Qwen, so a Qwen judge would rate its own family higher and quietly bias the comparison):

- **Mistral-Small-24B-Instruct**, ungated, no login needed
- **Prometheus 2** (7B), built specifically for rubric-based scoring

Judging is pairwise, not an absolute 1-10 score: for the same question, the judge is shown the baseline answer and a steered answer, and picks which one better presents multiple cultural viewpoints, A, B, or tie. Every pair is judged in both orders, so a judge that just follows position rather than content shows up as picking the same letter both times, which gets flagged automatically.

Aya Expanse 32B is also registered (key `"aya"`) as a stronger, multilingual-native alternative, but it's gated and needs a Hugging Face login, so it isn't the default here, swap it in once that access is set up.

In [ ]:
# pairs were built and judged per model inside the loop above.
# pairs_judged is the combined set across every model that's finished so far.
pairs_judged.head()

In [ ]:
# mistral judging already completed above, per model, with its own
# checkpointing. resolved_mistral here is the combined result.
print(f'{len(resolved_mistral):,} resolved mistral pairs across {resolved_mistral["model"].nunique()} models')

In [ ]:
# prometheus judging already completed above, per model, with its own
# checkpointing. resolved_prometheus here is the combined result.
print(f'{len(resolved_prometheus):,} resolved prometheus pairs across {resolved_prometheus["model"].nunique()} models')

In [ ]:
print("--- mistral win rates ---")
winrates_mistral = judge.summarise_pairwise(resolved_mistral)
display(winrates_mistral)

print("\n--- prometheus win rates ---")
winrates_prometheus = judge.summarise_pairwise(resolved_prometheus)
display(winrates_prometheus)

print("\n--- do the two judges agree? ---")
agreement = judge.judge_agreement(resolved_mistral, resolved_prometheus)

In [ ]:
# Point estimates alone cannot support the claim. A decisive win rate of
# 0.667 built on 9 decided pairs is not distinguishable from one of 0.200
# built on 5. These tests resample each condition and report an interval on
# the difference against its own random control.
print("--- does each method actually beat its random control? ---")
beat_mistral = judge.compare_to_random(resolved_mistral)
display(beat_mistral[beat_mistral['beats_random']])

print("
--- conditions that also beat the unsteered baseline ---")
print("(lower bound of the interval above 0.5, not just above random)")
display(winrates_mistral[winrates_mistral['beats_baseline']])

print("
Read n_decided, not n_pairs. High tie rates mean a condition with")
print("200 pairs can still rest on very few actual decisions.")

## Step 11b: a third judge, from a different family

Mistral-Small and Prometheus 2 both satisfy the rule that no judge comes from a family under test. They do not satisfy independence: Prometheus 2 is fine-tuned from Mistral-7B, so both sit on Mistral pretraining. Two judges from one base family plausibly share cultural blind spots, which is the exact failure this project is about, so their agreeing with each other is weaker evidence than it looks.

Llama 3.1 8B is a genuinely different lineage. The mirror used here is ungated, so it needs no licence acceptance or login, though the weights are still under Meta's Llama 3.1 Community Licence rather than an OSI-approved one, which makes them open-weights rather than open-source.

This runs on a subset. A robustness check does not need every pair, and a third full pass would cost about as much as the first two together. The subset is taken at pair level, never at row level: a pair split across the boundary loses one of its two orders, and `resolve_pairwise` treats a half pair as incomplete and quietly resolves it to a tie.

On kappa. Cohen's kappa is defined for exactly two raters, so a third judge does not remove it, it just needs the generalisation. Fleiss' kappa extends the same chance-correction to any fixed number of raters, and both are reported below: pairwise Cohen between each pair of judges, and one Fleiss figure across all three.

In [ ]:
# Subset at pair level so both presentation orders of a pair stay together.
judge_subset = judge.sample_pairs(pairs_judged, n_pairs=300)

judged_llama = judge.run_judge(judge_subset, "llama", tag='third_judge')
resolved_llama = judge.resolve_pairwise(judged_llama, tag='third_judge')
judge.unload_judge()

print("\n--- llama win rates ---")
display(judge.summarise_pairwise(resolved_llama))

resolved_all = {
    'mistral': resolved_mistral,
    'prometheus': resolved_prometheus,
    'llama': resolved_llama,
}

# Cohen's kappa pairwise, then Fleiss across all three at once.
agreement_matrix = judge.judge_agreement_matrix(resolved_all)
fleiss = judge.fleiss_kappa(resolved_all)

print('\nmistral vs prometheus share a base family, so their agreement is the one')
print('to discount. Either of them against llama is the real test of whether the')
print('result survives changing who is asked.')

## Step 12: human review (later)

Exports a blinded sample so a person can score it the same way the LLM judges did, for judge-vs-human agreement. Not meant to be filled in right now, this cell just produces the file to come back to.

Open `human_review_main.csv`, fill in the `winner` column with A, B, or tie for as many rows as you have time for, save it, then run the merge cell below.

In [ ]:
human_review = judge.export_for_human_review(pairs_judged, tag=run.tag)
human_review.head()

In [ ]:
# run this once human_review_main.csv has been filled in
import pandas as pd

human_csv_path = cfg.EVAL_DIR / f"human_review_{run.tag}.csv"
if human_csv_path.exists() and pd.read_csv(human_csv_path)["winner"].astype(str).str.strip().ne("").any():
    print("--- human vs mistral ---")
    judge.merge_human_review(human_csv_path, resolved_mistral, tag=run.tag)
    print("\n--- human vs prometheus ---")
    judge.merge_human_review(human_csv_path, resolved_prometheus, tag=run.tag)
else:
    print("human_review_main.csv has no filled-in verdicts yet, nothing to compare")

## Step 13: download the results

Everything is already saved to Drive as it runs, so this step is only for
pulling a copy onto the machine you are sitting at.

Zips `results/` and downloads it through the browser. The PRISM dataset is
left out: it is 128 MB and downloads itself from Hugging Face on any fresh
run, so there is no reason to carry it around.

In [ ]:
import os
import zipfile

archive_path = '/content/miorpa_results.zip'

with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for folder, _, filenames in os.walk('results'):
        for filename in filenames:
            file_path = os.path.join(folder, filename)
            archive.write(file_path, file_path)

size_mb = os.path.getsize(archive_path) / 1e6
n_files = len(zipfile.ZipFile(archive_path).namelist())
print(f'{n_files:,} files, {size_mb:.1f} MB -> {archive_path}')

try:
    from google.colab import files

    files.download(archive_path)
except ImportError:
    print('not on Colab; the zip is at the path above')